# TRG-Diffusion v13: thermal-guided VIIS fine-tuning (no zero-shot targets)
**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = Files (optional).
**Attach these inputs** (Add Input): `viis-master` (VIIS `last.ckpt`), `msrs-main` (MSRS), `llvip-dataset` (LLVIP), and the dataset you created from **VIIS_patched_v13.zip**.
Run top to bottom. Interactive: cells 1-5, then 6-8 when training is done. Unattended: cells 1-2, then cell **4b** only, and use *Save Version -> Save & Run All*.

In [ ]:
# 1. setup: copy the repo to a writable folder and install deps (re-run after every session restart)
import glob, os, shutil, subprocess, sys, pathlib
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "omegaconf", "kornia", "lpips", "piq", "ultralytics"])
try: import pytorch_lightning
except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning"])
cands = [os.path.dirname(p) for pat in ("/kaggle/input/*/train_v13.py", "/kaggle/input/*/*/train_v13.py",
         "/kaggle/input/*/*/*/train_v13.py", "/kaggle/input/*/*/*/*/train_v13.py") for p in glob.glob(pat)]
print("repo candidates:", cands); assert cands, "attach the VIIS_patched_v13 dataset"
REPO = "/kaggle/working/VIIS13"
if os.path.exists(REPO): shutil.rmtree(REPO)
shutil.copytree(cands[0], REPO, ignore=shutil.ignore_patterns("__pycache__"))
os.chdir(REPO); print("repo ->", REPO)

In [ ]:
# 2. checks: numpy unit tests (degradation, alignment, metrics, data splits), dataset/ckpt discovery, GPUs
!cd /kaggle/working/VIIS13 && python tests/test_v13_numpy.py | tail -n 3 && python -m trg13.paths
!nvidia-smi --query-gpu=name,memory.total --format=csv
# If a path shows NOT FOUND, set it, e.g.:  os.environ["LLVIP"] = "/kaggle/input/<...>/LLVIP"   (also MSRS, VIIS_CKPT)

In [ ]:
# 3. SMOKE TEST (~5-10 min): every torch path (adapter, losses, night branch, EMA, ckpt, tiled inference, metrics)
!cd /kaggle/working/VIIS13 && bash tests/smoke_kaggle.sh 2>&1 | grep -vi warn | tail -n 40

In [ ]:
# 4a. INTERACTIVE: launch both trainings in the background (GPU0 full model, GPU1 ablation without thermal losses)
import subprocess, time
R = "/kaggle/working/runs"; os.makedirs(R, exist_ok=True)
for gpu, name, extra in [(0, "v13_full", ""), (1, "v13_no_tc", "--thermal_loss 0")]:
    subprocess.Popen(f"cd /kaggle/working/VIIS13 && CUDA_VISIBLE_DEVICES={gpu} nohup python -u train_v13.py --name {name} {extra} "
                     f"> {R}/{name}.log 2>&1 &", shell=True)
    time.sleep(90)          # stagger: both read the 9 GB VIIS checkpoint
print("started; monitor with cell 5")

In [ ]:
# 4b. UNATTENDED alternative (use INSTEAD of 4a, e.g. in 'Save & Run All'): train both, evaluate, audit -- blocks until done
# !cd /kaggle/working/VIIS13 && bash run_all_v13.sh

In [ ]:
# 5. monitor: evaluation lines (psnr/ssim/lpips on MSRS val with real GT, night guards), last train line
!for n in v13_full v13_no_tc; do echo "== $n"; grep -E "EVAL|NEW BEST|STOP|DONE|Error|Traceback" /kaggle/working/runs/$n.log | cut -c1-420 | tail -n 6; tail -n 1 /kaggle/working/runs/$n.log | cut -c1-300; done
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv
from IPython.display import Image, display
f = sorted(glob.glob("/kaggle/working/runs/v13_full/prev_*.png"))
if f: print(f[-1]); display(Image(f[-1]))

In [ ]:
# 6. INFERENCE + EVALUATION (after both runs print DONE): VIIS zero-shot vs v13_no_tc vs v13 on
#    MSRS test (synthetic light/low/dark, full reference), MSRS night + LLVIP test (real, no reference, person boxes),
#    real light / low-light / dark samples.  Add your own pairs with --samples_dir <dir with rgb/ and ir/>.
!cd /kaggle/working/VIIS13 && python -u infer_v13.py --ckpts /kaggle/working/runs/v13_no_tc/best.pt,/kaggle/working/runs/v13_full/best.pt \
     --labels v13_no_tc,v13 --out /kaggle/working/infer_v13 2>&1 | grep -vi warn | tail -n 60

In [ ]:
# 7. alignment audit (deliverable 1, CPU) + look at the results
!cd /kaggle/working/VIIS13 && python align_audit_v13.py --out /kaggle/working/align_audit | tail -n 30
for p in sorted(glob.glob("/kaggle/working/infer_v13/grid_*.png")) + ["/kaggle/working/align_audit/overlay.png"]:
    print(p); display(Image(p))

In [ ]:
# 8. download bundle (checkpoints are large: best.pt ~0.6 GB each; latest.pt also holds the optimizer)
import shutil
for d in ("infer_v13", "align_audit"):
    if os.path.exists(f"/kaggle/working/{d}"): shutil.make_archive(f"/kaggle/working/{d}", "zip", f"/kaggle/working/{d}")
os.makedirs("/kaggle/working/runs_light", exist_ok=True)
for n in ("v13_full", "v13_no_tc"):
    s = f"/kaggle/working/runs/{n}"
    if os.path.isdir(s):
        shutil.copytree(s, f"/kaggle/working/runs_light/{n}", dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pt"))
shutil.make_archive("/kaggle/working/runs_light", "zip", "/kaggle/working/runs_light")
print(os.listdir("/kaggle/working"))